<a href="https://colab.research.google.com/github/AllenGeorge08/NeuralNetworksFromScratch/blob/master/Makemore3E02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [91]:
g = torch.Generator().manual_seed(2147483647)

In [101]:
# 100 samples
# 10 input features
# 3 output classes
X = torch.randn(100, 10)
Y = torch.randint(0, 3, (100,))

In [92]:
class Linear:
  def __init__(self,fan_in,fan_out,bias=True):
    self.weight = torch.randn((fan_in,fan_out),generator=g)/fan_in**0.5
    self.bias = torch.zeros(fan_out) if bias else None

  def __call__(self,x):
    self.out = x@self.weight
    if self.bias is not None:
      self.out += self.bias
    return self.out

  def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])



In [93]:
class BatchNorm1d:
  def __init__(self,dim,eps=1e-5,momentum=0.1):
    self.eps = eps
    self.momentum = momentum
    self.training = True
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

    self.running_mean = torch.zeros(dim)
    self.running_var = torch.ones(dim)


  def __call__(self,x):
    if self.training:
      xmean = x.mean(0,keepdim=True)
      xvar = x.var(0,keepdim=True)

      with torch.no_grad():
        self.running_mean = ((1-self.momentum)*self.running_mean + self.momentum*xmean)
        self.running_var = ((1-self.momentum)*self.running_var + self.momentum*xvar)

    else:
      xmean = self.running_mean
      xvar = self.running_var


    x_hat = (x-xmean)/torch.sqrt(xvar+self.eps)
    self.out = self.gamma*x_hat + self.beta
    return self.out


  def parameters(self):
    return [self.gamma,self.beta]



In [94]:
class Tanh:
  def __call__(self,x):
    self.out = torch.tanh(x)
    if self.out.requires_grad:
      self.out.retain_grad() #retain output of intermediate tensor
    return self.out


  def parameters(self):
    return []

In [98]:
layers = [
    Linear(10, 20),
    BatchNorm1d(20),
    Tanh(),

    Linear(20, 20),
    BatchNorm1d(20),
    Tanh(),

    Linear(20, 3)
]

In [103]:
parameters = []

for layer in layers:
    parameters += layer.parameters()

print("number of parameters:", len(parameters))
for p in parameters:
    p.requires_grad = True

number of parameters: 10


In [104]:
for step in range(1000):

    # forward
    x = X

    for layer in layers:
        x = layer(x)

    logits = x

    # loss
    loss = F.cross_entropy(logits, Y)

    # backward
    for p in parameters:
        p.grad = None

    loss.backward()

    # update
    lr = 0.1

    with torch.no_grad():
        for p in parameters:
            p -= lr * p.grad

    if step % 100 == 0:
        print(step, loss.item())

0 1.208186388015747
100 0.8777817487716675
200 0.476799875497818
300 0.2751864492893219
400 0.034543056041002274
500 0.01861269399523735
600 0.012621516361832619
700 0.009479464031755924
800 0.007517334073781967
900 0.00619091559201479


In [106]:
for layer in layers:

    if isinstance(layer, BatchNorm1d):
        layer.training = False

In [107]:
x = X

for layer in layers:
    x = layer(x)

original_logits = x
original_loss = F.cross_entropy(original_logits, Y)
print("original loss:", original_loss.item())

original loss: 0.005248703993856907


In [108]:
class LinearNew:
  def __init__(self,W,b):
    self.weight = W
    self.bias = b

  def __call__(self,x):
    self.out = x@self.weight + self.bias
    return self.out

  def parameters(self):
    return [self.weight,self.bias]



def fold_linear_bn(linear,bn):
  scale = bn.gamma/torch.sqrt(bn.running_var+bn.eps)
  new_weight = linear.weight * scale
  new_bias = (bn.beta+(linear.bias - bn.running_mean)*scale)

  return new_weight,new_bias


In [109]:
folded_layers = []

i = 0

while i < len(layers):

    if (
        isinstance(layers[i], Linear)
        and i + 1 < len(layers)
        and isinstance(layers[i + 1], BatchNorm1d)
    ):

        linear = layers[i]
        bn = layers[i + 1]

        new_weight, new_bias = fold_linear_bn(
            linear,
            bn
        )

        folded_layers.append(
            LinearNew(new_weight, new_bias)
        )

        i += 2

    else:

        folded_layers.append(layers[i])

        i += 1

In [110]:
x = X

for layer in folded_layers:
    x = layer(x)

folded_logits = x

In [113]:
folded_loss = F.cross_entropy(
    folded_logits,
    Y
)

print("original loss:", original_loss.item())
print("folded loss:", folded_loss.item())
difference = (
    original_logits - folded_logits
).abs()

print("maximum difference:", difference.max().item())

original loss: 0.005248703993856907
folded loss: 0.005248703993856907
maximum difference: 3.814697265625e-06
